In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [2]:
df = pd.read_csv("D:\DA-project-01-sales-analysis\da-project-01-sales-analysis\data\SuperStoreOrders.csv")
df.head()

<>:1: SyntaxWarning: invalid escape sequence '\D'
<>:1: SyntaxWarning: invalid escape sequence '\D'
C:\Users\U s e r\AppData\Local\Temp\ipykernel_15576\771038665.py:1: SyntaxWarning: invalid escape sequence '\D'
  df = pd.read_csv("D:\DA-project-01-sales-analysis\da-project-01-sales-analysis\data\SuperStoreOrders.csv")


,order_id,order_date,ship_date,ship_mode,customer_name,segment,state,country,market,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit,shipping_cost,order_priority,year
0,AG-2011-2040,1/1/2011,6/1/2011,Standard Class,Toby Braunhardt,Consumer,Constantine,Algeria,Africa,Africa,OFF-TEN-10000025,Office Supplies,Storage,"Tenex Lockers, Blue",408,2,0.00,106.14,35.46,Medium,2011
1,IN-2011-47883,1/1/2011,8/1/2011,Standard Class,Joseph Holt,Consumer,New South Wales,Australia,APAC,Oceania,OFF-SU-10000618,Office Supplies,Supplies,"Acme Trimmer, High Speed",120,3,0.10,36.04,9.72,Medium,2011
2,HU-2011-1220,1/1/2011,5/1/2011,Second Class,Annie Thurman,Consumer,Budapest,Hungary,EMEA,EMEA,OFF-TEN-10001585,Office Supplies,Storage,"Tenex Box, Single Width",66,4,0.00,29.64,8.17,High,2011
3,IT-2011-3647632,1/1/2011,5/1/2011,Second Class,Eugene Moren,Home Office,Stockholm,Sweden,EU,North,OFF-PA-10001492,Office Supplies,Paper,"Enermax Note Cards, Premium",45,3,0.50,-26.05,4.82,High,2011
4,IN-2011-47883,1/1/2011,8/1/2011,Standard Class,Joseph Holt,Consumer,New South Wales,Australia,APAC,Oceania,FUR-FU-10003447,Furniture,Furnishings,"Eldon Light Bulb, Duo Pack",114,5,0.10,37.77,4.70,Medium,2011


In [3]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

df.columns.tolist()

['order_id',
 'order_date',
 'ship_date',
 'ship_mode',
 'customer_name',
 'segment',
 'state',
 'country',
 'market',
 'region',
 'product_id',
 'category',
 'sub_category',
 'product_name',
 'sales',
 'quantity',
 'discount',
 'profit',
 'shipping_cost',
 'order_priority',
 'year']

## Missing values

In [4]:
missing_count=df.isnull().sum()
missing_pct= (df.isnull().sum()/len(df))*100

missing_summary=pd.DataFrame({
"missing_count":missing_count,
"missing_pct":missing_pct
}).sort_values("missing_count",ascending=False)

missing_summary

,missing_count,missing_pct
order_id,0,0.00
order_date,0,0.00
ship_date,0,0.00
ship_mode,0,0.00
customer_name,0,0.00
segment,0,0.00
state,0,0.00
country,0,0.00
market,0,0.00
region,0,0.00


## Duplicate Data

In [5]:
full_duplicate_count = df.duplicated().sum()
print("Full duplicate rows:", full_duplicate_count)

Full duplicate rows: 0


In [7]:
if "order_id" in df.columns:
    print("Duplicated order_id count:", df["order_id"].duplicated().sum())
    df["order_id"].value_counts().head(10)

Duplicated order_id count: 26255


## Datatype fix

In [8]:
date_cols = [c for c in df.columns if "date" in c]
date_cols

['order_date', 'ship_date']

In [9]:
for col in date_cols:
    df[col] = pd.to_datetime(
        df[col],
        format="mixed",
        dayfirst=True,
        errors="coerce"
    )

df[date_cols].dtypes

order_date    datetime64[ns]
ship_date     datetime64[ns]
dtype: object

In [10]:
for col in date_cols:
    print(f"{col}: {df[col].isna().sum()} missing values after conversion")

order_date: 0 missing values after conversion
ship_date: 0 missing values after conversion


In [11]:
numeric_candidates = ["sales", "profit", "discount", "quantity"]

for col in numeric_candidates:
    if col in df.columns:
        print(col, df[col].dtype)

sales object
profit float64
discount float64
quantity int64


In [13]:
for col in numeric_candidates:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df[numeric_candidates].dtypes

sales       float64
profit      float64
discount    float64
quantity      int64
dtype: object

## Validity checks

In [14]:
checks = {}

if "sales" in df.columns:
    checks["sales_negative"] = (df["sales"] < 0).sum()

if "quantity" in df.columns:
    checks["quantity_zero_or_negative"] = (df["quantity"] <= 0).sum()

if "discount" in df.columns:
    checks["discount_below_0"] = (df["discount"] < 0).sum()
    checks["discount_above_1"] = (df["discount"] > 1).sum()

if "ship_date" in df.columns and "order_date" in df.columns:
    checks["ship_before_order"] = (df["ship_date"] < df["order_date"]).sum()

pd.Series(checks)

sales_negative               0
quantity_zero_or_negative    0
discount_below_0             0
discount_above_1             0
ship_before_order            0
dtype: int64

## Suspicious values

In [16]:
df[["sales", "profit", "discount", "quantity"]].describe().T

,count,mean,std,min,25%,50%,75%,max
sales,"48,660.00",161.02,201.09,0.00,29.00,77.00,208.00,999.00
profit,"51,290.00",28.64,174.42,"-6,599.98",0.00,9.24,36.81,"8,399.98"
discount,"51,290.00",0.14,0.21,0.00,0.00,0.00,0.20,0.85
quantity,"51,290.00",3.48,2.28,1.00,2.00,3.00,5.00,14.00


In [17]:
high_discount_df = df[df["discount"] >= 0.5].copy()
print("Rows with discount >= 0.5:", len(high_discount_df))
high_discount_df.head(10)

Rows with discount >= 0.5: 5805


,order_id,order_date,ship_date,ship_mode,customer_name,segment,state,country,market,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit,shipping_cost,order_priority,year
3,IT-2011-3647632,2011-01-01,2011-01-05,Second Class,Eugene Moren,Home Office,Stockholm,Sweden,EU,North,OFF-PA-10001492,Office Supplies,Paper,"Enermax Note Cards, Premium",45.00,3,0.50,-26.05,4.82,High,2011
11,IN-2011-65159,2011-01-03,2011-01-07,Second Class,Larry Blacks,Consumer,National Capital,Philippines,APAC,Southeast Asia,FUR-TA-10002797,Furniture,Tables,"Chromcraft Round Table, Adjustable Height",211.00,1,0.55,-70.40,21.32,High,2011
33,US-2011-136007,2011-01-04,2011-01-11,Standard Class,Beth Thompson,Home Office,Alagoas,Brazil,LATAM,South,OFF-EN-10004956,Office Supplies,Envelopes,"Jiffy Interoffice Envelope, Set of 50",74.00,6,0.60,-107.86,7.04,Medium,2011
37,IT-2011-2942451,2011-01-04,2011-01-09,Standard Class,Grant Thornton,Corporate,England,United Kingdom,EU,North,OFF-AR-10002485,Office Supplies,Art,"Boston Markers, Easy-Erase",27.00,2,0.50,-21.90,2.11,Medium,2011
40,IT-2011-2942451,2011-01-04,2011-01-09,Standard Class,Grant Thornton,Corporate,England,United Kingdom,EU,North,OFF-ST-10001426,Office Supplies,Storage,"Eldon Folders, Single Width",17.00,2,0.50,-1.05,0.90,Medium,2011
49,CA-2011-112326,2011-01-05,2011-01-09,Standard Class,Phillina Ober,Home Office,Illinois,United States,US,Central,OFF-BI-10004094,Office Supplies,Binders,GBC Standard Plastic Binding Systems Combs,4.00,2,0.80,-5.49,0.55,High,2011
82,IT-2011-5134922,2011-01-07,2011-01-10,First Class,Joy Smith,Consumer,Groningen,Netherlands,EU,Central,OFF-AR-10000316,Office Supplies,Art,"Stanley Pens, Blue",5.00,1,0.50,-0.10,0.28,Medium,2011
85,IT-2011-4546695,2011-01-08,2011-01-14,Standard Class,Darren Powers,Consumer,Midi-Pyrénées,France,EU,Central,FUR-BO-10003103,Furniture,Bookcases,"Ikea Classic Bookcase, Metal",987.00,6,0.60,"-1,011.64",65.64,Medium,2011
87,CA-2011-105417,2011-01-08,2011-01-13,Standard Class,Vivek Sundaresam,Consumer,Texas,United States,US,Central,FUR-FU-10004864,Furniture,Furnishings,"Howard Miller 14-1/2"" Diameter Chrome Round Wa...",77.00,3,0.60,-53.71,6.69,Medium,2011
89,IT-2011-4546695,2011-01-08,2011-01-14,Standard Class,Darren Powers,Consumer,Midi-Pyrénées,France,EU,Central,OFF-AR-10000110,Office Supplies,Art,"Binney & Smith Sketch Pad, Blue",116.00,5,0.50,-55.65,3.91,Medium,2011


In [18]:
loss_df = df[df["profit"] < 0].copy()
print("Rows with negative profit:", len(loss_df))
loss_df.head(10)

Rows with negative profit: 12543


,order_id,order_date,ship_date,ship_mode,customer_name,segment,state,country,market,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit,shipping_cost,order_priority,year
3,IT-2011-3647632,2011-01-01,2011-01-05,Second Class,Eugene Moren,Home Office,Stockholm,Sweden,EU,North,OFF-PA-10001492,Office Supplies,Paper,"Enermax Note Cards, Premium",45.00,3,0.50,-26.05,4.82,High,2011
8,ID-2011-80230,2011-01-03,2011-01-09,Standard Class,Ken Lonsdale,Consumer,Auckland,New Zealand,APAC,Oceania,TEC-CO-10004182,Technology,Copiers,"Hewlett Wireless Fax, Laser",912.00,4,0.40,-319.46,107.10,Low,2011
10,IN-2011-65159,2011-01-03,2011-01-07,Second Class,Larry Blacks,Consumer,National Capital,Philippines,APAC,Southeast Asia,OFF-ST-10003020,Office Supplies,Storage,"Tenex Lockers, Industrial",338.00,3,0.45,-122.80,33.75,High,2011
11,IN-2011-65159,2011-01-03,2011-01-07,Second Class,Larry Blacks,Consumer,National Capital,Philippines,APAC,Southeast Asia,FUR-TA-10002797,Furniture,Tables,"Chromcraft Round Table, Adjustable Height",211.00,1,0.55,-70.40,21.32,High,2011
14,ID-2011-80230,2011-01-03,2011-01-09,Standard Class,Ken Lonsdale,Consumer,Auckland,New Zealand,APAC,Oceania,FUR-CH-10000214,Furniture,Chairs,"Hon Rocking Chair, Set of Two",159.00,2,0.40,-95.68,10.07,Low,2011
18,ID-2011-80230,2011-01-03,2011-01-09,Standard Class,Ken Lonsdale,Consumer,Auckland,New Zealand,APAC,Oceania,FUR-CH-10000666,Furniture,Chairs,"SAFCO Chairmat, Black",69.00,2,0.40,-26.41,8.17,Low,2011
19,ID-2011-12596,2011-01-03,2011-01-08,Standard Class,Chris McAfee,Consumer,Nakhon Ratchasima,Thailand,APAC,Southeast Asia,OFF-ST-10002066,Office Supplies,Storage,"Smead File Cart, Blue",135.00,2,0.47,-45.90,7.74,Medium,2011
33,US-2011-136007,2011-01-04,2011-01-11,Standard Class,Beth Thompson,Home Office,Alagoas,Brazil,LATAM,South,OFF-EN-10004956,Office Supplies,Envelopes,"Jiffy Interoffice Envelope, Set of 50",74.00,6,0.60,-107.86,7.04,Medium,2011
37,IT-2011-2942451,2011-01-04,2011-01-09,Standard Class,Grant Thornton,Corporate,England,United Kingdom,EU,North,OFF-AR-10002485,Office Supplies,Art,"Boston Markers, Easy-Erase",27.00,2,0.50,-21.90,2.11,Medium,2011
40,IT-2011-2942451,2011-01-04,2011-01-09,Standard Class,Grant Thornton,Corporate,England,United Kingdom,EU,North,OFF-ST-10001426,Office Supplies,Storage,"Eldon Folders, Single Width",17.00,2,0.50,-1.05,0.90,Medium,2011


## Text cleaning

In [19]:
text_cols = df.select_dtypes(include="object").columns.tolist()
text_cols

['order_id',
 'ship_mode',
 'customer_name',
 'segment',
 'state',
 'country',
 'market',
 'region',
 'product_id',
 'category',
 'sub_category',
 'product_name',
 'order_priority']

In [20]:
for col in text_cols:
    df[col] = df[col].apply(lambda x: x.strip() if isinstance(x, str) else x)

## Create a Cleaned Dataframe

In [21]:
df_clean = df.copy()

In [22]:
before_rows = len(df_clean)
df_clean = df_clean.drop_duplicates()
after_rows = len(df_clean)

print("Rows before dropping full duplicates:", before_rows)
print("Rows after dropping full duplicates:", after_rows)
print("Rows removed:", before_rows - after_rows)

Rows before dropping full duplicates: 51290
Rows after dropping full duplicates: 51290
Rows removed: 0


## Save cleaned data

In [23]:
import os
os.makedirs("../data/cleaned", exist_ok=True)

In [24]:
df_clean.to_csv("../data/cleaned/superstore_orders_cleaned.csv", index=False)